# Quickstart

For the purpose of this guide, let's assume we're working with sales data from two different time periods or systems, and we want to identify what has changed between them. As a running example, consider the following two data frames:

**Left (Baseline)**:
| `product_id` | `name` | `price` | `stock` | `category` |
|--------------|--------|---------|---------|------------|
| "A001" | "Widget" | 19.99 | 100 | "Tools" |
| "A002" | "Gadget" | 29.99 | 50 | "Electronics" |
| "A003" | "Doohickey" | 9.99 | 200 | "Tools" |

**Right (Current)**:
| `product_id` | `name` | `price` | `stock` | `category` |
|--------------|--------|---------|---------|------------|
| "A001" | "Widget" | 21.99 | 95 | "Tools" |
| "A002" | "Gadget" | 29.99 | 50 | "Electronics" |
| "A004" | "Thingamajig" | 14.99 | 75 | "Hardware" |

## Basic comparison

To get started with `diffly`, you'll use the {func}`~diffly.compare_frames` function to compare two Polars DataFrames:

In [1]:
import polars as pl
from diffly import compare_frames

left = pl.DataFrame({
    "product_id": ["A001", "A002", "A003"],
    "name": ["Widget", "Gadget", "Doohickey"],
    "price": [19.99, 29.99, 9.99],
    "stock": [100, 50, 200],
    "category": ["Tools", "Electronics", "Tools"]
})

right = pl.DataFrame({
    "product_id": ["A001", "A002", "A004"],
    "name": ["Widget", "Gadget", "Thingamajig"],
    "price": [21.99, 29.99, 14.99],
    "stock": [95, 50, 75],
    "category": ["Tools", "Electronics", "Hardware"]
})

# Compare the data frames
comparison = compare_frames(left, right, primary_key="product_id")

This creates a {class}`~diffly.DataFrameComparison` object that can be used to explore the differences between the two data frames.

## Checking equality

The simplest check is whether the two data frames are equal:

In [2]:
if comparison.equal():
    print("Data frames are identical!")
else:
    print("Data frames have differences")

Data frames have differences


In our example, this prints "Data frames have differences" because:

- Product A001's price changed from 19.99 to 21.99
- Product A001's stock changed from 100 to 95
- Product A003 exists only in the left data frame
- Product A004 exists only in the right data frame

## Generating a summary

To understand what changed, you can generate a detailed summary:

In [3]:
comparison.summary()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                     Diffly Summary                                     ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
   Primary key: product_id

 Schemas
 ▔▔▔▔▔▔▔
   Schemas match exactly (column count: 5).

 Rows
 ▔▔▔▔
   Left count             Right count
       3      (no change)      3

   ┏━┯━┯━┯━┯━┓
   ┃-│-│-│-│-┃                1  left only   (33.33%)
   ┠─┼─┼─┼─┼─┨╌╌╌┏━┯━┯━┯━┯━┓╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╮
   ┃ │ │ │ │ ┃ = ┃ │ │ │ │ ┃  1  equal       (50.00%)  │
   ┠─┼─┼─┼─┼─┨╌╌╌┠─┼─┼─┼─┼─┨╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌├╴  2  joined
   ┃ │ │ │ │ ┃ ≠ ┃ │ │ │ │ ┃  1  unequal     (50.00%)  │
   ┗━┷━┷━┷━┷━┛╌╌╌┠─┼─┼─┼─┼─┨╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╯
                 ┃+│+│+│+│+┃  1  right only  (33.33%)
                 ┗━┷━┷━┷━┷━┛

 Columns
 ▔▔▔▔▔▔▔
   ┌──────────┬─────────┐
   │ category │ 100.00% │
   │ name     │ 100.00% │
   │ p

This displays a rich, formatted summary showing:

- **Schema differences**: Columns that exist only in one data frame
- **Row differences**: Rows that exist only in one data frame (based on the primary key)
- **Column match rates**: What percentage of rows have matching values for each column

The summary provides a high-level overview of all differences in a human-readable format.

## Investigating differences

Beyond the summary, `diffly` provides methods to retrieve the actual rows that differ. For example, to get rows where `price` changed:

In [ ]:
comparison.joined_unequal("price")

## Working without a primary key

If your data frames don't have a natural primary key, you can still perform comparisons:

In [4]:
comparison_no_pk = compare_frames(left, right)  # No primary key specified

# Check if schemas match
if comparison_no_pk.equal():
    print("Data frames are identical")
else:
    print("Data frames have differences")

Data frames have differences


Without a primary key, diffly can only check:

- Schema equality (column names and types)
- Overall equality (including row order)

Row-level comparisons and detailed summaries require a primary key to join the data frames.

```{note}
`diffly` works seamlessly with both {class}`polars.DataFrame` and {class}`polars.LazyFrame`. Internally, it uses lazy evaluation for efficiency, only computing what's necessary when you request specific comparisons.
```

## Outlook

This concludes the quickstart guide. For more information, explore the [Features](features/index.md) section or see the [API Reference](../api/modules.rst) for all available comparison methods and configuration options.